# ۲. استخراج موج‌ها روی دیسک کولب

آرشیوها روی گوگل‌درایو می‌مانند. فایل‌ها در /content/aurora_waveforms نوشته می‌شوند، که دیسک موقت همین ماشین کولب است. با قطع شدن runtime این پوشه پاک می‌شود و استخراج را می‌توان دوباره اجرا کرد؛ فایلی که قبلاً کامل نوشته شده رد می‌شود.

اعداد مورد انتظار از فهرست قبلی:

- سمعی: ۱۱۴۲۷ فایل
- اسیلومتریک: ۲۷۱۹۶ فایل


In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('خارج از کولب؛ اتصال درایو رد شد.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import shutil
import zipfile
from pathlib import Path

DATASET_DIR = Path('/content/drive/MyDrive/aurora_dataset')
ZIPS = {
    'auscultatory': DATASET_DIR / 'measurements_auscultatory (1).zip',
    'oscillometric': DATASET_DIR / 'measurements_oscillometric.zip',
}
EXPECTED = {'auscultatory': 11427, 'oscillometric': 27196}
DEST = Path('/content/aurora_waveforms')
TABLES = [
    'participants.tsv',
    'measurements_auscultatory.tsv',
    'measurements_oscillometric.tsv',
    'features.tsv',
]

if 'MyDrive' in DEST.parts or 'drive' in DEST.parts:
    raise RuntimeError('مقصد نباید روی گوگل‌درایو باشد.')
free_gb = shutil.disk_usage('/content').free / (1024 ** 3)
print(f'فضای آزاد: {free_gb:.1f} GB')
if free_gb < 40:
    raise RuntimeError('فضای آزاد برای استخراج کافی نیست.')
for label, path in ZIPS.items():
    if not path.is_file():
        raise FileNotFoundError(path)
    print(f'{label}: {path.stat().st_size / (1024 ** 3):.2f} GB')


فضای آزاد: 107.5 GB
auscultatory: 1.51 GB
oscillometric: 4.67 GB


In [4]:
def extract_zip(path: Path, dest: Path, label: str) -> int:
    written = 0
    skipped = 0
    with zipfile.ZipFile(path) as archive:
        members = [info for info in archive.infolist() if not info.is_dir()]
        total = len(members)
        print(f'{label}: {total} فایل در آرشیو', flush=True)
        for index, info in enumerate(members, start=1):
            name = info.filename.replace('\\', '/').lstrip('/')
            if name.startswith('/') or '..' in Path(name).parts:
                raise RuntimeError(f'مسیر غیرمجاز در آرشیو: {name}')
            target = dest / name
            if target.exists() and target.stat().st_size == info.file_size:
                skipped += 1
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as source, target.open('wb') as output:
                    shutil.copyfileobj(source, output, length=1024 * 1024)
                written += 1
            if index % 1000 == 0 or index == total:
                print(f'{label}: {index}/{total}  نوشته‌شده={written}  موجود={skipped}', flush=True)
    return written

DEST.mkdir(parents=True, exist_ok=True)
for label, path in ZIPS.items():
    extract_zip(path, DEST, label)


auscultatory: 11427 فایل در آرشیو
auscultatory: 1000/11427  نوشته‌شده=1000  موجود=0
auscultatory: 2000/11427  نوشته‌شده=2000  موجود=0
auscultatory: 3000/11427  نوشته‌شده=3000  موجود=0
auscultatory: 4000/11427  نوشته‌شده=4000  موجود=0
auscultatory: 5000/11427  نوشته‌شده=5000  موجود=0
auscultatory: 6000/11427  نوشته‌شده=6000  موجود=0
auscultatory: 7000/11427  نوشته‌شده=7000  موجود=0
auscultatory: 8000/11427  نوشته‌شده=8000  موجود=0
auscultatory: 9000/11427  نوشته‌شده=9000  موجود=0
auscultatory: 10000/11427  نوشته‌شده=10000  موجود=0
auscultatory: 11000/11427  نوشته‌شده=11000  موجود=0
auscultatory: 11427/11427  نوشته‌شده=11427  موجود=0
oscillometric: 27196 فایل در آرشیو
oscillometric: 1000/27196  نوشته‌شده=1000  موجود=0
oscillometric: 2000/27196  نوشته‌شده=2000  موجود=0
oscillometric: 3000/27196  نوشته‌شده=3000  موجود=0
oscillometric: 4000/27196  نوشته‌شده=4000  موجود=0
oscillometric: 5000/27196  نوشته‌شده=5000  موجود=0
oscillometric: 6000/27196  نوشته‌شده=6000  موجود=0
oscillometric: 7000

In [5]:
import json
copied = []
for name in TABLES:
    source = DATASET_DIR / name
    if not source.is_file():
        print(f'جدول پیدا نشد: {name}')
        continue
    target = DEST / name
    shutil.copy2(source, target)
    copied.append(name)
print('جدول‌های کپی‌شده:', copied)

counts = {}
for label, folder in (
    ('auscultatory', DEST / 'measurements_auscultatory'),
    ('oscillometric', DEST / 'measurements_oscillometric'),
):
    found = sum(1 for path in folder.rglob('*.tsv') if path.is_file()) if folder.is_dir() else 0
    counts[label] = found
    print(f'{label}: {found} فایل، انتظار {EXPECTED[label]}')

report = {
    'extract_dir': str(DEST),
    'counts': counts,
    'expected': EXPECTED,
    'tables': copied,
    'matches_inventory': counts == EXPECTED,
}
report_path = Path('/content/aurorabp_outputs/extract_report.json')
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(report_path)
report


جدول‌های کپی‌شده: ['participants.tsv', 'measurements_auscultatory.tsv', 'measurements_oscillometric.tsv', 'features.tsv']
auscultatory: 11427 فایل، انتظار 11427
oscillometric: 27196 فایل، انتظار 27196
/content/aurorabp_outputs/extract_report.json


{'extract_dir': '/content/aurora_waveforms',
 'counts': {'auscultatory': 11427, 'oscillometric': 27196},
 'expected': {'auscultatory': 11427, 'oscillometric': 27196},
 'tables': ['participants.tsv',
  'measurements_auscultatory.tsv',
  'measurements_oscillometric.tsv',
  'features.tsv'],
 'matches_inventory': True}